<a href="https://colab.research.google.com/github/GilbertOwen/reefscan-id/blob/test-data/notebooks/08_test_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReefScan-ID: Stage 2 test data

Patches cut from the Coralscapes test split with exactly the settings of notebook 04. Nothing is evaluated here: no model is loaded and no metric is computed. The test split is used once, in E4 at the end. Run in Google Colab; no GPU is needed.

### Setup: Colab runtime and Google Drive

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
raw_dir = base_dir + "/RawData"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))
print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))
print("CPU count: {}".format(os.cpu_count()))

Mounted at /content/drive
Drive folder found: True
Cache folder exists: True
CPU count: 2


### Setup: libraries

In [2]:
!pip install -q datasets

import datasets
import numpy as np
import pandas as pd

print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))
print("pandas version: {}".format(pd.__version__))

datasets version: 4.8.5
numpy version: 2.4.6
pandas version: 2.2.3


## Section 1: Coralscapes test patches

Question: can the test photos be cut into stage 2 patches with the settings saved by notebook 04, and are the saved files complete and separate from the train and validation patches?

Checks: read the settings that notebook 04 saved, rebuild the label map, put the grid points on the test photos, apply the purity threshold and the cap, cut the 128 and 256 px patches, save them under new file names without touching any existing file, read them back, and check that no test photo or test patch is also in the train or validation patches. Only the test split is read, and nothing is evaluated.

### Load the dataset

In [3]:
import time
from datasets import load_dataset

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))
print("Test rows: {}, expected 392 as printed by notebook 07: {}".format(ds["test"].num_rows, ds["test"].num_rows == 392))

README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1517
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 166
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 392
    })
})
Loading took 47.1 seconds
Test rows: 392, expected 392 as printed by notebook 07: True


### Settings from settings.json

The values come from the file that notebook 04 saved. None of them is typed again here.

In [4]:
import json
import shutil

out_dir = base_dir + "/Processed/stage2_patches"
local_dir = "/content/stage2_patches_test"
os.makedirs(local_dir, exist_ok=True)

with open(out_dir + "/settings.json") as handle:
    settings = json.load(handle)

grid_cell = settings["grid_cell"]
margin = settings["margin"]
seed = settings["grid_seed"]
final_cap = settings["cap_per_state_per_photo"]
cap_seed = settings["cap_seed"]
final_threshold = settings["purity_threshold"]
threshold_crop = settings["purity_threshold_crop"]
save_crops = settings["crop_sizes_saved"]
state_names = {int(key): value for key, value in settings["state_names"].items()}
excluded_ids = settings["excluded_ids"]

print("Settings file: {}".format(out_dir + "/settings.json"))
print("Grid cell: {} px, margin: {} px, grid seed: {}".format(grid_cell, margin, seed))
print("Cap: {} points per state per photo, cap seed: {}".format(final_cap, cap_seed))
print("Purity threshold: {} measured at the {} px crop".format(final_threshold, threshold_crop))
print("Crop sizes saved: {}".format(save_crops))
print("State names: {}".format(state_names))
print("Excluded ids: {}".format(excluded_ids))
print("Splits cut by notebook 04: {}".format(settings["splits_cut"]))
print("Folder on Drive: {}".format(out_dir))
print("Folder on the Colab disk: {}".format(local_dir))

Settings file: /content/drive/MyDrive/ReefScan-ID/Processed/stage2_patches/settings.json
Grid cell: 128 px, margin: 128 px, grid seed: 0
Cap: 20 points per state per photo, cap seed: 0
Purity threshold: 0.7 measured at the 128 px crop
Crop sizes saved: [128, 256]
State names: {0: 'healthy', 1: 'bleached', 2: 'dead'}
Excluded ids: [21, 27]
Splits cut by notebook 04: ['train', 'validation']
Folder on Drive: /content/drive/MyDrive/ReefScan-ID/Processed/stage2_patches
Folder on the Colab disk: /content/stage2_patches_test


### Class names from the dataset card

Ids and names copied from notebook 04, which copied them from the table on the Coralscapes dataset card.

In [5]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

Classes on the card: 39


### Stage 2 label map

In [6]:
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]

excluded_state = 253
not_coral_state = 255

# Values that are not coral stay at the not coral code
state_map = np.full(256, not_coral_state, dtype=np.uint8)
state_map[healthy_ids] = 0
state_map[bleached_ids] = 1
state_map[dead_ids] = 2
state_map[excluded_ids] = excluded_state

for state_id, state_name in state_names.items():
    names = [class_names[v] for v in class_names if state_map[v] == state_id]
    print("{} ({} ids): {}".format(state_name, len(names), names))
excluded = [class_names[v] for v in class_names if state_map[v] == excluded_state]
print("excluded coral ({} ids): {}".format(len(excluded), excluded))
print("Ids that are not coral: {}".format(sum(1 for v in class_names if state_map[v] == not_coral_state)))

healthy (8 ids): ['other coral alive', 'massive/meandering alive', 'branching alive', 'acropora alive', 'table acropora alive', 'pocillopora alive', 'stylophora alive', 'meandering alive']
bleached (4 ids): ['other coral bleached', 'massive/meandering bleached', 'branching bleached', 'meandering bleached']
dead (5 ids): ['other coral dead', 'branching dead', 'massive/meandering dead', 'table acropora dead', 'meandering dead']
excluded coral (2 ids): ['millepora', 'turbinaria']
Ids that are not coral: 20


### Grid for the test photos

In [7]:
crop_sizes = save_crops
photo_height = 1024
photo_width = 2048
cell_rows = (photo_height - 2 * margin) // grid_cell
cell_cols = (photo_width - 2 * margin) // grid_cell

print("Grid cell: {} px, crop sizes: {}".format(grid_cell, crop_sizes))
print("Margin from the photo edge: {} px, half the largest crop: {}".format(margin, max(crop_sizes) // 2))
print("Grid cells per photo: {} rows x {} columns = {}".format(cell_rows, cell_cols, cell_rows * cell_cols))

Grid cell: 128 px, crop sizes: [128, 256]
Margin from the photo edge: 128 px, half the largest crop: 128
Grid cells per photo: 6 rows x 14 columns = 84


### Measure points on the test photos

In [8]:
def integral_table(flags):
    table = np.zeros((flags.shape[0] + 1, flags.shape[1] + 1), dtype=np.int64)
    table[1:, 1:] = flags.cumsum(axis=0).cumsum(axis=1)
    return table


def window_sum(table, tops, lefts, size):
    return (table[tops + size, lefts + size] - table[tops, lefts + size]
            - table[tops + size, lefts] + table[tops, lefts])


cell_count = cell_rows * cell_cols
records = []
for split in ["test"]:
    start = time.time()
    masks = ds[split].select_columns(["label"])
    for photo, row in enumerate(masks):
        state = state_map[np.array(row["label"])]

        # One random point in every grid cell
        rng = np.random.default_rng(seed + photo)
        ys = margin + np.repeat(np.arange(cell_rows), cell_cols) * grid_cell + rng.integers(0, grid_cell, cell_count)
        xs = margin + np.tile(np.arange(cell_cols), cell_rows) * grid_cell + rng.integers(0, grid_cell, cell_count)

        centre = state[ys, xs].astype(np.int64)
        keep = centre <= 2
        ys, xs, centre = ys[keep], xs[keep], centre[keep]
        if len(ys) == 0:
            continue

        tables = [integral_table(state == state_id) for state_id in state_names]
        record = {"split": split, "photo": photo, "y": ys, "x": xs, "state": centre}
        for size in crop_sizes:
            tops = ys - size // 2
            lefts = xs - size // 2
            sums = np.stack([window_sum(table, tops, lefts, size) for table in tables])
            coral = sums.sum(axis=0)
            same = sums[centre, np.arange(len(centre))]
            record["purity_{}".format(size)] = same / coral
            record["coverage_{}".format(size)] = coral / (size * size)
        records.append(pd.DataFrame(record))
    print("{}: {} photos measured in {:.1f} seconds".format(split, masks.num_rows, time.time() - start))

points = pd.concat(records, ignore_index=True)
points["state_name"] = points["state"].map(state_names)
print("Points kept: {}".format(len(points)))
print(points.head(5).to_string())

test: 392 photos measured in 74.9 seconds
Points kept: 5103
  split  photo    y     x  state  purity_128  coverage_128  purity_256  coverage_256 state_name
0  test      0  193   448      1    1.000000      0.705750    0.982506      0.670746   bleached
1  test      0  167   737      1    0.576944      0.569153    0.554515      0.796875   bleached
2  test      0  133   818      2    1.000000      0.758850    0.832671      0.764175       dead
3  test      0  137   938      1    0.605266      0.607361    0.425984      0.735352   bleached
4  test      0  130  1137      1    0.970235      0.732056    0.921793      0.788239   bleached


### Select the points

In [9]:
shuffled = points.sample(frac=1, random_state=cap_seed)

# Threshold first, then the cap on the points that remain
passed = shuffled[shuffled["purity_{}".format(threshold_crop)] >= final_threshold]
order = passed.groupby(["split", "photo", "state_name"]).cumcount()
selected = passed[order < final_cap].sort_values(["split", "photo", "y", "x"]).reset_index(drop=True)

selected["source"] = "coralscapes"
selected["photo_id"] = selected["split"] + "_" + selected["photo"].astype(str).str.zfill(4)
selected["row"] = selected.groupby("split").cumcount()
selected["label"] = selected["state_name"]

print("Points selected: {}".format(len(selected)))
print(pd.crosstab(selected["label"], selected["split"]).to_string())

Points selected: 3956
split     test
label         
bleached   284
dead       746
healthy   2926


### Cut the patches

In [10]:
from numpy.lib.format import open_memmap

for split in ["test"]:
    start = time.time()
    chosen = selected[selected["split"] == split]

    # Arrays are written to the Colab disk first
    arrays = {}
    for size in save_crops:
        array_path = "{}/coralscapes_{}_{}.npy".format(local_dir, split, size)
        arrays[size] = open_memmap(array_path, mode="w+", dtype=np.uint8, shape=(len(chosen), size, size, 3))

    images = ds[split].select_columns(["image"])
    for photo, group in chosen.groupby("photo"):
        image = np.array(images[int(photo)]["image"])
        for row, y, x in zip(group["row"], group["y"], group["x"]):
            for size in save_crops:
                half = size // 2
                arrays[size][row] = image[y - half:y + half, x - half:x + half]

    for size in save_crops:
        arrays[size].flush()
        print("{} {} px: array shape {}, dtype {}".format(split, size, arrays[size].shape, arrays[size].dtype))
    print("{}: {} photos cut in {:.1f} seconds".format(split, chosen["photo"].nunique(), time.time() - start))

test 128 px: array shape (3956, 128, 128, 3), dtype uint8
test 256 px: array shape (3956, 256, 256, 3), dtype uint8
test: 328 photos cut in 42.1 seconds


### Save the test patches to Drive

Four new file names are used. The cell refuses to save if one of them exists, and the files are opened in a mode that cannot overwrite, so no existing file in the folder is changed.

In [11]:
manifest_columns = ["source", "split", "photo_id", "photo", "row", "y", "x", "label", "state",
                    "purity_128", "coverage_128", "purity_256", "coverage_256"]
selected[manifest_columns].to_csv(local_dir + "/coralscapes_test_manifest.csv", index=False)

test_settings = dict(settings)
test_settings["splits_cut"] = ["test"]
with open(local_dir + "/settings_test.json", "w") as handle:
    json.dump(test_settings, handle, indent=2)

new_names = ["coralscapes_test_128.npy", "coralscapes_test_256.npy",
             "coralscapes_test_manifest.csv", "settings_test.json"]
print("Files built on the Colab disk: {}".format(sorted(os.listdir(local_dir))))

before = sorted(os.listdir(out_dir))
before_stats = {name: (os.path.getsize(out_dir + "/" + name), os.path.getmtime(out_dir + "/" + name)) for name in before}
print("Files in the folder before saving ({}): {}".format(len(before), before))

existing = [name for name in new_names if name in before]
print("New file names that already exist in the folder: {}".format(existing))
if len(existing) > 0:
    print("NOT SAVED: a new file name already exists, nothing was copied")
else:
    for name in new_names:
        start = time.time()
        # Mode xb refuses to open a file that already exists
        with open(local_dir + "/" + name, "rb") as source_file, open(out_dir + "/" + name, "xb") as target_file:
            shutil.copyfileobj(source_file, target_file)
        print("{}: {:.1f} MB, copied in {:.1f} seconds".format(
            name, os.path.getsize(out_dir + "/" + name) / 1024 ** 2, time.time() - start))

after = sorted(os.listdir(out_dir))
print("Files in the folder after saving ({}): {}".format(len(after), after))
print("Files added: {}".format([name for name in after if name not in before]))
print("Files removed: {}".format([name for name in before if name not in after]))
print("Size and modified time of every old file unchanged: {}".format(
    all(name in after and before_stats[name] == (os.path.getsize(out_dir + "/" + name), os.path.getmtime(out_dir + "/" + name))
        for name in before)))

Files built on the Colab disk: ['coralscapes_test_128.npy', 'coralscapes_test_256.npy', 'coralscapes_test_manifest.csv', 'settings_test.json']
Files in the folder before saving (9): ['coralscapes_manifest.csv', 'coralscapes_train_128.npy', 'coralscapes_train_256.npy', 'coralscapes_validation_128.npy', 'coralscapes_validation_256.npy', 'noaa_manifest.csv', 'noaa_train_224.npy', 'noaa_validation_224.npy', 'settings.json']
New file names that already exist in the folder: []
coralscapes_test_128.npy: 185.4 MB, copied in 0.9 seconds
coralscapes_test_256.npy: 741.8 MB, copied in 6.0 seconds
coralscapes_test_manifest.csv: 0.4 MB, copied in 0.2 seconds
settings_test.json: 0.0 MB, copied in 0.0 seconds
Files in the folder after saving (13): ['coralscapes_manifest.csv', 'coralscapes_test_128.npy', 'coralscapes_test_256.npy', 'coralscapes_test_manifest.csv', 'coralscapes_train_128.npy', 'coralscapes_train_256.npy', 'coralscapes_validation_128.npy', 'coralscapes_validation_256.npy', 'noaa_manifest

### Read back check

In [12]:
test_manifest = pd.read_csv(out_dir + "/coralscapes_test_manifest.csv")
old_manifest = pd.read_csv(out_dir + "/coralscapes_manifest.csv")
print("Test manifest columns equal those of coralscapes_manifest.csv: {}".format(
    list(test_manifest.columns) == list(old_manifest.columns)))
print("Splits in the test manifest: {}".format(sorted(test_manifest["split"].unique())))

saved = {}
for size in save_crops:
    saved[size] = np.load("{}/coralscapes_test_{}.npy".format(out_dir, size), mmap_mode="r")
    print("test {} px: array shape {}, dtype {}".format(size, saved[size].shape, saved[size].dtype))
    print("test {} px: {} patches in the file, {} rows in the manifest".format(
        size, saved[size].shape[0], len(test_manifest)))

print("Test photos with at least one patch: {} of {}".format(test_manifest["photo"].nunique(), ds["test"].num_rows))
print("Patches per state on test:")
for state_id, state_name in state_names.items():
    print("  {}: {}".format(state_name, int((test_manifest["state"] == state_id).sum())))

# A sample of rows is compared with the mask and with the larger patch
sample = test_manifest.sample(min(20, len(test_manifest)), random_state=0)
label_matches = 0
crop_matches = 0
for item in sample.itertuples():
    mask = np.array(ds["test"].select_columns(["label"])[int(item.photo)]["label"])
    label_matches += int(state_map[mask[item.y, item.x]] == item.state)
    small = saved[128][item.row]
    large = saved[256][item.row]
    crop_matches += int(np.array_equal(large[64:192, 64:192], small))
print("Centre label matches the mask: {} of {} sampled rows".format(label_matches, len(sample)))
print("128 px patch equals the centre of the 256 px patch: {} of {} sampled rows".format(crop_matches, len(sample)))

Test manifest columns equal those of coralscapes_manifest.csv: True
Splits in the test manifest: ['test']
test 128 px: array shape (3956, 128, 128, 3), dtype uint8
test 128 px: 3956 patches in the file, 3956 rows in the manifest
test 256 px: array shape (3956, 256, 256, 3), dtype uint8
test 256 px: 3956 patches in the file, 3956 rows in the manifest
Test photos with at least one patch: 328 of 392
Patches per state on test:
  healthy: 2926
  bleached: 284
  dead: 746
Centre label matches the mask: 20 of 20 sampled rows
128 px patch equals the centre of the 256 px patch: 20 of 20 sampled rows


### Test photo ids against the train and validation manifest

In [13]:
shared_ids = set(test_manifest["photo_id"]) & set(old_manifest["photo_id"])
id_prefixes = sorted(set(old_manifest["photo_id"].str.rsplit("_", n=1).str[0]))

print("Test photo ids: {}, photo ids in coralscapes_manifest.csv: {}".format(
    test_manifest["photo_id"].nunique(), old_manifest["photo_id"].nunique()))
print("Test photo ids that appear in coralscapes_manifest.csv: {}".format(len(shared_ids)))
print("Prefixes of the photo ids in coralscapes_manifest.csv: {}".format(id_prefixes))

Test photo ids: 328, photo ids in coralscapes_manifest.csv: 1530
Test photo ids that appear in coralscapes_manifest.csv: 0
Prefixes of the photo ids in coralscapes_manifest.csv: ['train', 'validation']


### Test patches against train and validation patches

The photo ids contain the split name, so the check above is true by construction. This check compares pixels: it looks for a test patch (128 px) that is byte-identical to a train or validation patch.

In [14]:
import hashlib


def patch_hashes(array):
    return [hashlib.md5(np.ascontiguousarray(array[index]).tobytes()).hexdigest() for index in range(len(array))]


test_hashes = patch_hashes(saved[128])
print("Test patches: {}, distinct among them: {}".format(len(test_hashes), len(set(test_hashes))))

for split in ["train", "validation"]:
    start = time.time()
    other = np.load("{}/coralscapes_{}_128.npy".format(out_dir, split), mmap_mode="r")
    other_hashes = set(patch_hashes(other))
    print("{}: {} patches hashed in {:.1f} seconds, test patches identical to one of them: {}".format(
        split, len(other), time.time() - start, sum(1 for value in test_hashes if value in other_hashes)))

Test patches: 3956, distinct among them: 3956
train: 22253 patches hashed in 15.4 seconds, test patches identical to one of them: 0
validation: 2320 patches hashed in 5.1 seconds, test patches identical to one of them: 0


**Observation**: From the 392 Coralscapes test photos, we selected 3,956 patches from 328 photos using the same settings as training. These include 2,926 healthy, 284 bleached, and 746 dead patches. All saved patches matched the manifest, and no identical patches were found in training or validation. However, we haven't evaluated the model yet, so these results only confirm the test data preparation.

## Section 2: NOAA test manifest and array

Question: can the NOAA test images be listed in a manifest and packed into one array the same way as the train and validation images, and how much do their file names overlap with the train and validation file names?

Checks: list the test images of the Drive copy with the label from the class folder, compare the counts with the notebook 03 check, read the images into one array, save the manifest and the array under new file names without touching any existing file, read them back and compare a sample with the original files, then count the file names and the name parts that test shares with train and validation. Only the NOAA test folder and the existing NOAA manifest are read, and nothing is evaluated.

### Settings for the NOAA test images

The state ids come from `settings.json`, as in Section 1. The counts to compare with are the ones of the notebook 03 check.

In [ ]:
from numpy.lib.format import open_memmap
from PIL import Image

noaa_dir = raw_dir + "/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
noaa_labels = {"CORAL": "healthy", "CORAL_BL": "bleached"}
noaa_splits = {"test": "test"}
image_endings = (".jpg", ".jpeg", ".png")
state_ids = {name: state_id for state_id, name in state_names.items()}
patch_size = 224
expected_counts = {"CORAL": 974, "CORAL_BL": 591}

print("NOAA folder: {}".format(noaa_dir))
print("Folders in the NOAA folder: {}".format(sorted(os.listdir(noaa_dir))))
print("Class folders in the test folder: {}".format(sorted(os.listdir(noaa_dir + "/test"))))
print("Labels and state ids: {}".format({label: state_ids[label] for label in noaa_labels.values()}))
print("Image size in the array: {} by {}".format(patch_size, patch_size))

### NOAA test manifest

In [ ]:
rows = []
for folder, split in noaa_splits.items():
    for class_folder, label in noaa_labels.items():
        for name in sorted(os.listdir("{}/{}/{}".format(noaa_dir, folder, class_folder))):
            if name.lower().endswith(image_endings):
                rows.append({
                    "source": "noaa",
                    "split": split,
                    "photo_id": "{}/{}/{}".format(folder, class_folder, name),
                    "label": label,
                    "state": state_ids[label],
                })

noaa_manifest = pd.DataFrame(rows)
print("NOAA rows: {}".format(len(noaa_manifest)))
print(pd.crosstab(noaa_manifest["label"], noaa_manifest["split"]).to_string())

### NOAA test counts against notebook 03

In [ ]:
for class_folder, expected_count in expected_counts.items():
    found = int((noaa_manifest["photo_id"].str.split("/").str[1] == class_folder).sum())
    print("{}: {} images, expected {}, same: {}".format(class_folder, found, expected_count, found == expected_count))
print("Total: {} images, expected {}, same: {}".format(
    len(noaa_manifest), sum(expected_counts.values()), len(noaa_manifest) == sum(expected_counts.values())))
print("Example photo ids: {}".format(list(noaa_manifest["photo_id"].head(3))))

### Read the NOAA test images into an array

In [ ]:
from concurrent.futures import ThreadPoolExecutor

read_threads = 16
noaa_manifest["row"] = noaa_manifest.groupby("split").cumcount()


def read_noaa(photo_id):
    with Image.open(noaa_dir + "/" + photo_id) as image:
        found = "{}x{} {}".format(image.size[1], image.size[0], image.mode)
        picture = np.array(image.convert("RGB").resize((patch_size, patch_size), Image.BILINEAR))
    return picture, found


expected = "{}x{} RGB".format(patch_size, patch_size)
for split in ["test"]:
    start = time.time()
    rows = noaa_manifest[noaa_manifest["split"] == split]
    array_path = "{}/noaa_{}_{}.npy".format(local_dir, split, patch_size)
    array = open_memmap(array_path, mode="w+", dtype=np.uint8, shape=(len(rows), patch_size, patch_size, 3))

    # Files are read in parallel threads and written in manifest order
    odd = []
    with ThreadPoolExecutor(max_workers=read_threads) as pool:
        for row, (picture, found) in enumerate(pool.map(read_noaa, rows["photo_id"])):
            array[row] = picture
            if found != expected:
                odd.append((rows["photo_id"].iloc[row], found))
    array.flush()

    print("{}: array shape {}, read in {:.1f} seconds".format(split, array.shape, time.time() - start))
    print("{}: images that were not {}: {}".format(split, expected, len(odd)))
    for photo_id, found in odd[:20]:
        print("  {}: {}".format(photo_id, found))

### Save the NOAA test files to Drive

Two new file names are used. The cell refuses to save if one of them exists, and the files are opened in a mode that cannot overwrite, so no existing file in the folder is changed. `noaa_manifest.csv` is not touched.

In [ ]:
noaa_columns = ["source", "split", "photo_id", "label", "state", "row"]
noaa_manifest[noaa_columns].to_csv(local_dir + "/noaa_test_manifest.csv", index=False)

noaa_new_names = ["noaa_test_224.npy", "noaa_test_manifest.csv"]
print("New files built on the Colab disk: {}".format([name for name in noaa_new_names if os.path.isfile(local_dir + "/" + name)]))

before = sorted(os.listdir(out_dir))
before_stats = {name: (os.path.getsize(out_dir + "/" + name), os.path.getmtime(out_dir + "/" + name)) for name in before}
print("Files in the folder before saving ({}): {}".format(len(before), before))

existing = [name for name in noaa_new_names if name in before]
print("New file names that already exist in the folder: {}".format(existing))
if len(existing) > 0:
    print("NOT SAVED: a new file name already exists, nothing was copied")
else:
    for name in noaa_new_names:
        start = time.time()
        # Mode xb refuses to open a file that already exists
        with open(local_dir + "/" + name, "rb") as source_file, open(out_dir + "/" + name, "xb") as target_file:
            shutil.copyfileobj(source_file, target_file)
        print("{}: {:.1f} MB, copied in {:.1f} seconds".format(
            name, os.path.getsize(out_dir + "/" + name) / 1024 ** 2, time.time() - start))

after = sorted(os.listdir(out_dir))
print("Files in the folder after saving ({}): {}".format(len(after), after))
print("Files added: {}".format([name for name in after if name not in before]))
print("Files removed: {}".format([name for name in before if name not in after]))
print("Size and modified time of every old file unchanged: {}".format(
    all(name in after and before_stats[name] == (os.path.getsize(out_dir + "/" + name), os.path.getmtime(out_dir + "/" + name))
        for name in before)))

### Read back check

In [ ]:
noaa_test_manifest = pd.read_csv(out_dir + "/noaa_test_manifest.csv")
old_noaa_manifest = pd.read_csv(out_dir + "/noaa_manifest.csv")
print("Test manifest columns equal those of noaa_manifest.csv: {}".format(
    list(noaa_test_manifest.columns) == list(old_noaa_manifest.columns)))
print("Splits in the test manifest: {}".format(sorted(noaa_test_manifest["split"].unique())))
print("Splits in noaa_manifest.csv, which was not changed: {}".format(sorted(old_noaa_manifest["split"].unique())))

packed_test = np.load("{}/noaa_test_{}.npy".format(out_dir, patch_size), mmap_mode="r")
print("Array shape {}, dtype {}".format(packed_test.shape, packed_test.dtype))
print("{} images in the array, {} rows in the manifest, equal: {}".format(
    packed_test.shape[0], len(noaa_test_manifest), packed_test.shape[0] == len(noaa_test_manifest)))
print("Row numbers run from 0 to {} in order: {}".format(
    len(noaa_test_manifest) - 1, list(noaa_test_manifest["row"]) == list(range(len(noaa_test_manifest)))))
print("Images per label: {}".format(noaa_test_manifest["label"].value_counts().to_dict()))

sample = noaa_test_manifest.sample(min(20, len(noaa_test_manifest)), random_state=0)
same = 0
for item in sample.itertuples():
    with Image.open(noaa_dir + "/" + item.photo_id) as image:
        original = np.array(image.convert("RGB"))
    same += int(np.array_equal(packed_test[item.row], original))
print("Array rows identical to the original file: {} of {} sampled rows".format(same, len(sample)))

### NOAA test file names against train and validation

In [ ]:
old_noaa_manifest["file_name"] = old_noaa_manifest["photo_id"].str.split("/").str[-1]
test_names = noaa_test_manifest["photo_id"].str.split("/").str[-1]
train_names = set(old_noaa_manifest.loc[old_noaa_manifest["split"] == "train", "file_name"])
val_names = set(old_noaa_manifest.loc[old_noaa_manifest["split"] == "validation", "file_name"])

print("File names: {} in test, {} in train, {} in validation".format(test_names.nunique(), len(train_names), len(val_names)))
print("Test file names repeated inside test: {}".format(int(test_names.duplicated().sum())))
print("Test file names that also appear in train: {}".format(int(test_names.isin(train_names).sum())))
print("Test file names that also appear in validation: {}".format(int(test_names.isin(val_names).sum())))
print("Test file names that also appear in train or validation: {}".format(int(test_names.isin(train_names | val_names).sum())))

### Overlap of NOAA name parts between test and train

In [ ]:
def name_parts(table):
    stems = table["photo_id"].str.split("/").str[-1].str.rsplit(".", n=1).str[0]
    return stems.str.split("_")


def name_groups(table):
    parts = name_parts(table)
    return pd.DataFrame({"first_part": parts.str[0], "first_three": parts.str[:3].str.join("_")})


print("Number of parts per test name: {}".format(name_parts(noaa_test_manifest).str.len().value_counts().to_dict()))

test_groups = name_groups(noaa_test_manifest)
train_groups = name_groups(old_noaa_manifest[old_noaa_manifest["split"] == "train"])

for column, label in [("first_part", "first part (site)"), ("first_three", "first three parts")]:
    shared = set(test_groups[column]) & set(train_groups[column])
    in_shared = int(test_groups[column].isin(shared).sum())
    print("{}: {} groups in test, {} in train, {} in both".format(
        label, test_groups[column].nunique(), train_groups[column].nunique(), len(shared)))
    print("{}: {} of {} test images are in a group that also has train images".format(
        label, in_shared, len(test_groups)))

**Observation**: